# Lesson 4 · Stationary waves and harmonics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/04_stationary_waves.ipynb)

**For:** A-level Physics (for example AQA 3.3.1.3, OCR A 4.4.4, Edexcel Topic 5) and IB Physics (C.4 standing waves and resonance). Check your own specification for the exact wording.

**Before this lesson:** Lesson 1, *Simple harmonic motion* (oscillations, amplitude and phase). Also assumed: the wave equation v = f lambda.

**Leads to:** Lesson 7, *Wave–particle duality and the double slit*; Lesson 15, *Diffraction gratings and optics*.

**You will be able to:**

1. explain how a stationary wave forms from two progressive waves, and identify nodes and antinodes
2. use $f_n = \dfrac{n}{2L}\sqrt{T/\mu}$ for a stretched string
3. state the phase relationships between points on a stationary wave, and how it differs from a progressive wave
4. find the resonant frequencies of open and closed pipes
5. explain why a plucked string produces several harmonics, and which ones are missing

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import waves

## 1. Two travelling waves make a stationary wave

When two waves of the same frequency and amplitude travel in **opposite directions** through each other, they superpose to give

$$y = A\sin(kx - \omega t) + A\sin(kx + \omega t) = 2A\sin(kx)\cos(\omega t).$$

The shape $\sin kx$ no longer moves; it just grows and shrinks. Points where $\sin kx = 0$ never move at all: these are **nodes**. Halfway between them are **antinodes**, which oscillate with the largest amplitude, $2A$.

In [ ]:
x = np.linspace(0, 2, 400)
k, w, A = 2 * np.pi, 2 * np.pi, 1.0           # wavelength 1, period 1
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, t in zip(axes, [0, 0.1, 0.2, 0.25]):
    right = A * np.sin(k * x - w * t)
    left = A * np.sin(k * x + w * t)
    ax.plot(x, right, color="C0", lw=1, alpha=0.6, label="to the right")
    ax.plot(x, left, color="C1", lw=1, alpha=0.6, label="to the left")
    ax.plot(x, right + left, "k", lw=2, label="sum")
    ax.plot([0, 0.5, 1, 1.5, 2], [0] * 5, "o", color="C3", ms=5)
    ax.set_title(f"t = {t} T"); ax.set_xlabel("x / wavelength")
axes[0].legend(fontsize=7, loc="lower left");

The red dots are the nodes: half a wavelength apart, and still at every instant. At $t = T/4$ the two travelling waves cancel everywhere and the string is momentarily flat (but moving fastest).

**Stationary vs progressive.** In a stationary wave, energy is *not* transferred along the wave. All points between neighbouring nodes move **in phase**; points either side of a node are in **antiphase**.

## 2. Harmonics on a string

A string fixed at both ends must have a node at each end, so only certain wavelengths fit: a whole number of half-wavelengths, $L = n\lambda_n/2$. With wave speed $v = \sqrt{T/\mu}$ (tension $T$, mass per unit length $\mu$),

$$f_n = \frac{n v}{2L} = \frac{n}{2L}\sqrt{\frac{T}{\mu}},\qquad n = 1, 2, 3, \dots$$

$n = 1$ is the **fundamental** (first harmonic). Below, the wave equation $u_{tt} = c^2 u_{xx}$ is solved step by step, starting from the shape of each harmonic, and the period of each is measured.

In [ ]:
L, c = 1.0, 1.0
x = np.linspace(0, L, 201)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, n in zip(axes, [1, 2, 3]):
    t, u = waves.wave_1d(np.sin(n * np.pi * x / L), x, c=c, t_max=10 * L / c)
    one_period = np.searchsorted(t, 2 * L / (n * c))
    for frame in np.linspace(0, one_period, 9).astype(int):
        ax.plot(x, u[frame], color="C0", alpha=0.5, lw=1)
    antinode = np.argmin(np.abs(x - L / (2 * n)))
    s = u[:, antinode]
    up = np.where((s[:-1] < 0) & (s[1:] >= 0))[0]                     # upward zero crossings...
    crossings = t[up] - s[up] * (t[up + 1] - t[up]) / (s[up + 1] - s[up])   # ...interpolated between steps
    period = np.diff(crossings).mean()
    ax.set_title(f"n = {n}: measured T = {period:.4f}, 2L/(nc) = {2 * L / (n * c):.4f}", fontsize=9)
    ax.set_xlabel("x / m")

Each harmonic oscillates in place at exactly $f_n = nc/2L$.

## 3. A plucked string plays many harmonics at once

A real pluck is not a pure harmonic: it is a triangle. That shape is a **sum** of harmonics, so the string vibrates at all of them at once, and you hear the mixture as the instrument's tone.

The pluck point starts displaced, so it cannot be a node of any harmonic that is present: a harmonic with a node exactly at the pluck point is never excited, and is **missing** from the sound. Below we pluck the simulated string, record one point over time, and take the spectrum.

**Predict (P1).** The string is plucked one fifth of the way along. Using the rule above, which harmonics will be missing from the spectrum? Write your prediction down before you run the next cell.

In [ ]:
L, c, pluck_at = 1.0, 1.0, 0.2                 # pluck at one fifth of the length   <- Try it: 0.25, 0.5
x = np.linspace(0, L, 401)
triangle = np.where(x < pluck_at, x / pluck_at, (L - x) / (L - pluck_at)) * 0.01
t, u = waves.wave_1d(triangle, x, c=c, t_max=40)
freqs, amps = waves.spectrum(u[:, np.argmin(np.abs(x - 0.13))], t[1] - t[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
for frame in range(0, 400, 50):
    ax1.plot(x, u[frame], lw=1)
ax1.set(xlabel="x / m", title=f"Plucked at x = {pluck_at} L")
f1 = c / (2 * L)
ax2.plot(freqs / f1, amps)
ax2.set(xlim=(0, 16), xlabel="frequency / fundamental frequency", ylabel="amplitude", title="Spectrum of the motion at one point")
ax2.set_xticks(range(0, 17))
for n in range(1, 16):
    if np.isclose(np.sin(n * np.pi * pluck_at / L), 0, atol=1e-9):
        ax2.axvline(n, color="C3", ls=":", lw=1)

**Explain (P1).** Why is the 5th harmonic the first to vanish, and which others go with it?

The peaks are at whole multiples of the fundamental, and the red dotted lines mark the harmonics with a node at the pluck point: they are missing. This is why guitarists pluck near the bridge for a bright, harmonic-rich sound and near the middle for a mellow one (plucking at the middle removes every even harmonic).

## 4. Pipes

In a pipe, the air vibrates. A **closed** end must be a displacement **node**; an **open** end is an **antinode**.

- **Open at both ends:** antinode at each end, $L = n\lambda/2$, so $f_n = \dfrac{nv}{2L}$: all harmonics.
- **Closed at one end:** node at one end and antinode at the other, $L = (2n-1)\lambda/4$, so $f = \dfrac{v}{4L}, \dfrac{3v}{4L}, \dfrac{5v}{4L}, \dots$: only **odd** harmonics.

In [ ]:
v_sound, L = 340.0, 0.50                       # m/s, m
xs = np.linspace(0, L, 200)
fig, axes = plt.subplots(2, 3, figsize=(11, 4), sharex=True, sharey=True)
for j in range(3):
    n_open, n_closed = j + 1, 2 * j + 1
    disp_open = np.cos(n_open * np.pi * xs / L)                     # antinodes at both ends
    disp_closed = np.sin(n_closed * np.pi * xs / (2 * L))           # node at x = 0 (closed), antinode at L
    for ax, d, f, label in [(axes[0, j], disp_open, n_open * v_sound / (2 * L), "open-open"),
                            (axes[1, j], disp_closed, n_closed * v_sound / (4 * L), "closed-open")]:
        ax.fill_between(xs, d, -d, alpha=0.25); ax.plot(xs, d, "C0"); ax.plot(xs, -d, "C0")
        ax.set_title(f"{label}: {f:.0f} Hz", fontsize=9); ax.set_yticks([])
axes[1, 0].set_xlabel("x / m  (closed end at x = 0)");

**Exam tip:** "first overtone" means the next frequency above the fundamental. For a closed pipe the first overtone is the **third** harmonic, $3v/4L$, because the even ones do not exist.

## 5. Any repeating wave is a sum of sine waves

Section 3 is one example of a general result, **Fourier's theorem**: any periodic signal can be built from sine waves at the fundamental frequency and its harmonics,

$$f(t) = \frac{a_0}{2} + \sum_{n=1}^{\infty}\left(a_n\cos n\omega t + b_n\sin n\omega t\right).$$

Below, the package computes the series for a square wave.

**Predict (P2).** Does a square wave need odd harmonics only, even harmonics only, or both? Hint: shift a square wave by half a period and you get the same wave upside down. Write your prediction down before you run the next cell.

In [ ]:
square = lambda s: 1.0 if (s % 1.0) < 0.5 else -1.0
a0, a, b = waves.fourier_coefficients(square, 1.0, 15, points=[0.5])
print("b_n (n = 1..9):", np.round(b[:9], 4))
print("4/(pi n)      :", np.round([4 / (np.pi * n) if n % 2 else 0 for n in range(1, 10)], 4))
ts = np.linspace(0, 2, 1000)
plt.figure(figsize=(8, 3.4))
plt.plot(ts, [square(s) for s in ts], "k", lw=1, label="square wave")
for N in (1, 3, 15):                          # <- Try it: more terms
    plt.plot(ts, waves.fourier_sum(a0, a[:N], b[:N], ts, 1.0), label=f"{N} terms")
plt.legend(fontsize=8, loc="upper right"); plt.xlabel("t / period");

**Explain (P2).** Why does that half-period symmetry rule out the even harmonics?

Adding harmonics sharpens the corners. (The small overshoot at each jump never disappears however many terms you add: it is called the Gibbs phenomenon.)

---

## Exercises

Give answers to 3 significant figures.

**Q1.** A guitar string is 0.65 m long, has mass per unit length $4.0\times10^{-4}\ \mathrm{kg\,m^{-1}}$ and is under a tension of 60 N. Find (a) the wave speed on the string, (b) the fundamental frequency, (c) the frequency of the third harmonic.

**Q2.** What tension would tune the string in Q1 to a fundamental of 330 Hz?

**Q3.** A pipe 0.30 m long is closed at one end. The speed of sound is 340 m s$^{-1}$. Find (a) its fundamental frequency, (b) the frequency of its first overtone.

**Q4.** In a stationary sound wave of frequency 950 Hz, adjacent nodes are 0.18 m apart. Find the speed of sound.

**Q5 (simulation).** A string is plucked at a quarter of its length. Which is the lowest harmonic that is missing from its sound? Check your answer by changing `pluck_at` in section 3.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_v1 = np.sqrt(60 / 4.0e-4)
_missing = next(n for n in range(1, 50) if np.isclose(np.sin(n * np.pi * 0.25), 0, atol=1e-9))
_answers = {
    "Q1a": _v1, "Q1b": _v1 / (2 * 0.65), "Q1c": 3 * _v1 / (2 * 0.65),
    "Q2": 4.0e-4 * (2 * 0.65 * 330) ** 2,
    "Q3a": 340 / (4 * 0.30), "Q3b": 3 * 340 / (4 * 0.30),
    "Q4": 2 * 0.18 * 950,
    "Q5": _missing,
}

# Replace None with your answers, in SI units (Q5 as the harmonic number).
mine = {"Q1a": None, "Q1b": None, "Q1c": None, "Q2": None, "Q3a": None, "Q3b": None, "Q4": None, "Q5": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** (a) $v = \sqrt{T/\mu} = \sqrt{60/4.0\times10^{-4}} = 387\ \mathrm{m\,s^{-1}}$ &nbsp; (b) $f_1 = v/2L = 387.3/1.30 = 298\ \mathrm{Hz}$ &nbsp; (c) $f_3 = 3f_1 = 894\ \mathrm{Hz}$

**Q2.** $f_1 = \dfrac{1}{2L}\sqrt{\dfrac{T}{\mu}} \Rightarrow T = \mu(2Lf_1)^2 = 4.0\times10^{-4} \times (2 \times 0.65 \times 330)^2 = 73.6\ \mathrm{N}$

**Q3.** (a) $f_1 = v/4L = 340/1.20 = 283\ \mathrm{Hz}$ &nbsp; (b) The first overtone of a closed pipe is the third harmonic: $3 \times 283.3 = 850\ \mathrm{Hz}$.

**Q4.** Adjacent nodes are half a wavelength apart, so $\lambda = 0.36$ m and $v = f\lambda = 950 \times 0.36 = 342\ \mathrm{m\,s^{-1}}$.

**Q5.** A harmonic is missing if it has a node at the pluck point, i.e. if $\sin(n\pi \times \tfrac14) = 0$. The first $n$ for which that happens is $n = 4$ (then 8, 12, ...).

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")

**Next:** in section 3, set `pluck_at = 0.5`. Which harmonics disappear now, and why does the string's shape stay symmetric?

### Predict and explain: answers

**P1.** The 5th, 10th, 15th, ... The $n$th harmonic has nodes at $x = L/n, 2L/n, \dots$, so the pluck point $x = L/5$ is a node exactly when $n$ is a multiple of 5. Those harmonics would need the string to stay still where it was pulled, so they are never excited.

**P2.** Odd harmonics only, with amplitudes $4/\pi n$ (the printed $b_n$ match). Shifting by half a period multiplies the $n$th harmonic by $(-1)^n$: odd harmonics flip sign, even ones do not. A square wave flips sign under that shift, so it can only contain harmonics that flip too, the odd ones.